# Schur Complement Lens

Chapter 10: Matrix Block Partitioning · *Mathematics for AI and Machine Learning* by Fuheng Wu

Inspect how eliminating one block yields Schur complement updates.

Companion code for the [interactive demo](https://math4ai.org/companion/chapter-10/schur-complement/) on math4ai.org (sign-in required). Run the cell with Shift+Enter; PyTorch is preinstalled on Colab. Change the values at the top and run it again to explore.


In [1]:
import torch

torch.manual_seed(0)

# The demo's default scalar blocks of M = [[A, B], [C, D]]
A, B, C, D = 1.6, 0.9, -0.4, 1.3
S = D - C * B / A                              # Schur complement of A
print("Schur complement S = D - C A^-1 B =", S)
print("det M =", A * D - B * C, " det(A) * S =", A * S)

# ── Matrix blocks: eliminating the first block row leaves S ─────────────────
n, m = 3, 2
M = torch.randn(n + m, n + m, dtype=torch.float64) + 3 * torch.eye(n + m, dtype=torch.float64)
A_, B_, C_, D_ = M[:n, :n], M[:n, n:], M[n:, :n], M[n:, n:]
S_ = D_ - C_ @ torch.linalg.solve(A_, B_)

# Block LU: M = [[I, 0], [C A^-1, I]] [[A, B], [0, S]]
L = torch.eye(n + m, dtype=M.dtype)
L[n:, :n] = C_ @ torch.linalg.inv(A_)
U = torch.zeros_like(M)
U[:n, :n], U[:n, n:], U[n:, n:] = A_, B_, S_
print("\nM == L U (block LU):", torch.allclose(L @ U, M))
print("det M == det A * det S:", torch.linalg.det(M).item(), torch.linalg.det(A_).item() * torch.linalg.det(S_).item())

# The bottom-right block of M^{-1} is S^{-1}
print("(M^-1)_22 == S^-1:", torch.allclose(torch.linalg.inv(M)[n:, n:], torch.linalg.inv(S_)))

# ── In statistics: conditioning a Gaussian uses the same S ──────────────────
X = torch.randn(n + m, n + m, dtype=torch.float64)
Sigma = X @ X.T + torch.eye(n + m, dtype=X.dtype)
S11, S12, S21, S22 = Sigma[:n, :n], Sigma[:n, n:], Sigma[n:, :n], Sigma[n:, n:]
cond_cov = S22 - S21 @ torch.linalg.solve(S11, S12)
print("\nCov(x2 | x1) = Schur complement, inverse of the precision block:",
      torch.allclose(cond_cov, torch.linalg.inv(torch.linalg.inv(Sigma)[n:, n:])))

Schur complement S = D - C A^-1 B = 1.5250000000000001
det M = 2.44  det(A) * S = 2.4400000000000004

M == L U (block LU): True
det M == det A * det S: 10.306050426268214 10.306050426268213
(M^-1)_22 == S^-1: True

Cov(x2 | x1) = Schur complement, inverse of the precision block: True
